In [ ]:
import fiona
import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs

In [ ]:
#list of files that you wish to combine onto one map
files = [
"/path/to/shapefile1.shp",
"/path/to/shapefile2.shp"
]

In [ ]:
#get contours and height of those contours from the files
contours = []
values = []
for f in files:
    with fiona.open(f,"r") as shapefile:
        for feature in shapefile:
            contours.append(feature["geometry"])
            values.append(feature['properties']['PROP_VALUE'])

In [ ]:
#set up colourmap for the contours, here we scale the contours so higher elevations are lighter in colour.

cmap = 'Oranges_r'
min_c = min(values)
max_c = max(values)

min_y = 0
max_y = 0.6

gradient = (max_y-min_y)/(max_c-min_c)
intercept = max_y - gradient*max_c

In [ ]:
def set_size(w,h, ax=None):
    #helper function to set size of map area
    #from https://stackoverflow.com/questions/44970010/axes-class-set-explicitly-size-width-height-of-axes-in-given-units
    """ w, h: width, height in inches """
    if not ax: ax=plt.gca()
    l = ax.figure.subplotpars.left
    r = ax.figure.subplotpars.right
    t = ax.figure.subplotpars.top
    b = ax.figure.subplotpars.bottom
    figw = float(w)/(r-l)
    figh = float(h)/(t-b)
    ax.figure.set_size_inches(figw, figh)

In [ ]:
def calculate_dims(extent,scale):
    #helper function to scale produced map to desired scale. Takes map extent and desired scale 1:x and returns size of map in inches.
    x0,x1 = extent[0],extent[1]
    y0,y1 = extent[2],extent[3]

    scaled_x = ((100*(x1-x0))/scale)/2.54
    scaled_y = ((100*(y1-y0))/scale)/2.54

    return scaled_x,scaled_y

In [ ]:
#make plot

fig = plt.figure()
ax = fig.add_subplot(111,projection=ccrs.OSGB())

#get xy coordinates of every contour line and extent of full map
i=0
min_x_coord = np.inf
max_x_coord = 0
min_y_coord = np.inf
max_y_coord = 0
for contour in contours:
    #print(contour)
    xs = []
    ys = []
    pair = 0
    while pair<len(contour['coordinates']):
        xs.append(contour['coordinates'][pair][0])
        ys.append(contour['coordinates'][pair][1])
        min_x_coord = min(min_x_coord, contour['coordinates'][pair][0])
        max_x_coord = max(max_x_coord, contour['coordinates'][pair][0])
        min_y_coord = min(min_y_coord, contour['coordinates'][pair][1])
        max_y_coord = max(max_y_coord, contour['coordinates'][pair][1])
       
        pair = pair+1
    #plot each contour

    # This is the line that actually plots contours.
    # The colour 'c' is set by scaling the height in metres to the range found above.
    # The line width 'lw' can be adjusted, increase the number if you want wider contours, decrease for thinner.
    ax.plot(xs,ys,c=plt.get_cmap(cmap)(values[i]*gradient+intercept),lw=0.1)
    
    i=i+1

#set up x y ticks and labels
xticks = np.arange(min_x_coord,max_x_coord+1000,1000)  #tick every 1000m 
ax.set_xticks(xticks)
xlabels = [(tick%100000)/1000 for tick in xticks]
xlabels = ["{:02d}".format(int(number)) for number in xlabels]
ax.set_xticklabels(xlabels)

yticks = np.arange(min_y_coord,max_y_coord+1000,1000)
ax.set_yticks(yticks)
ylabels = [(tick%100000)/1000 for tick in yticks]
ylabels = ["{:02d}".format(int(number)) for number in ylabels]
ax.set_yticklabels(ylabels)
ax.tick_params('x', top=True, labeltop=True)
ax.tick_params('y', right=True, labelright=True)

#plot gridlines
for yn in yticks:
    ax.plot([min_x_coord,max_x_coord],[yn,yn],c=(0,183/255,241/255),alpha=.5,lw=.1)
for xn in xticks:
    ax.plot([xn,xn],[min_y_coord,max_y_coord],c=(0,183/255,241/255),alpha=.5,lw=.1)

#set map extent. This is custom and should be of form [x0,x1,y0,y1]. In OS grid coordinates, including leading digit.
#e.g for Ben Nevis at NN 167 712 we convert the letters to their easting and northing: these are normally shown in smaller type on maps as a prefix to the grid square every 10km or so.
#We also need to change to a 10 figure grid ref (plus leading prefix): for Ben Nevis we arrive at 216700 771200, which is contained in the extent below.
extent = [214200,218200,770800,773200]
ax.set_extent(extent,crs=ccrs.OSGB())

#add annotation and plot title
ax.annotate("Contours at 10m intervals. Map scale 1:15000 when printed actual size A4. \nContains public sector information licensed under the Open Government Licence v3.0.", (0.005,-0.1),xycoords='axes fraction')
ax.set_title('Ben Nevis OS Terrain 50 contours')

#calculate size of map using our extent and our desired scale (1:15000)
x_size,y_size = calculate_dims(extent,15000)

#This is where you control what scale and size your map is. Values are in **inches**: calculate how big you want your map size to be and pass those values.
#For example 10km by 10km at 1:25000 is 40cm by 40cm, which is 15.748" by 15.748".
set_size(x_size, y_size)
plt.savefig('map.pdf',bbox_inches='tight')